In [ ]:
import os
import iso639
import openai

openai.api_base = os.environ["OPENAI_API_BASE"]
openai.api_key = os.environ["OPENAI_API_KEY"]

MODEL = "gpt-4"


In [ ]:
# Helper functions

def get_language_name(language_code: str) -> str:
    """
    Example:
    >>> get_language_name("en")
    'English' 
    """
    return iso639.to_name(language_code)


def get_prompt(sentence: str, prompt_template=None, lang_code=None):
    if prompt_template is None:
        prompt_template = "Correct the OCRed {language} sentence \"{sentence}\""

    if lang_code is None:
        lang_code = "en"

    return prompt_template.format(language=get_language_name(lang_code), sentence=sentence.strip())


def generate_output(sentence: str, lang_code: str, prompt_template: str):
    """
    Temperature is set to 0 to avoid randomization and be deterministic. 
    """
    lang = get_language_name(lang_code)
    prompt = prompt_template.format(language=lang, sentence=sentence.strip())

    response = openai.ChatCompletion.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0
                                            )
    corrected = response.choices[0]["message"]["content"]

    return corrected


def generate_outputs(l_sentences: list[str], lang_code: str, prompt_template):
    for sentence in l_sentences:
        yield generate_output(sentence, lang_code, prompt_template)

In [ ]:
prompt_template = "Correct the OCRed {language} sentence \"{sentence}\""

lang_code = "en"

text = [
    "Thise is a sente with speling mistkez",
    "Once upon a time",
    "in a land far far away",
    "there was a princess."
]

for corrected in generate_outputs(text, lang_code, prompt_template):
    print(corrected)
    print(" ")


## Find better prompts etc

In [ ]:
prompt_template = "Correct the OCRed {language} text lines \"{sentence}\". Return a JSON with 'text' field."

for corrected in generate_outputs(text, lang_code, prompt_template):
    print(corrected)
    print(" ")

In [ ]:
prompt_template = "Correct the OCRed text in {language} \"{sentence}\". Only correct character errors. Return a JSON with 'text' field."

for corrected in generate_outputs(text, lang_code, prompt_template):
    print(corrected)
    print(" ")